In [3]:
from googlenewsdecoder import new_decoderv1
import time

df_valid = pd.read_csv("hasil_scraping_isi_artikel.csv")

# Test 5 URL berturut-turut dengan jeda 3 detik
for i in range(5):
    url = df_valid.loc[i, "URL"]
    result = new_decoderv1(url)
    print(f"URL {i}: {result['status']} | {result.get('decoded_url', result.get('message', ''))[:80]}")
    time.sleep(3)

URL 0: False | Request error in decode_url: 429 Client Error: Too Many Requests for url: https:
URL 1: False | Request error in decode_url: 429 Client Error: Too Many Requests for url: https:
URL 2: False | Request error in decode_url: 429 Client Error: Too Many Requests for url: https:
URL 3: False | Request error in decode_url: 429 Client Error: Too Many Requests for url: https:
URL 4: False | Request error in decode_url: 429 Client Error: Too Many Requests for url: https:


In [1]:
import pandas as pd
import time
import os
from newspaper import Article
from concurrent.futures import ThreadPoolExecutor, as_completed
from googlenewsdecoder import new_decoderv1

# =========================
# 1. LOAD DATA
# =========================
if os.path.exists("hasil_scraping_isi_artikel.csv"):
    df_valid = pd.read_csv("hasil_scraping_isi_artikel.csv")
    df_valid["Actual_URL"] = df_valid["Actual_URL"].replace("None", pd.NA)
    df_valid["Content"] = df_valid["Content"].replace("None", pd.NA)
    print("Resume dari file sebelumnya...")
else:
    df_all = pd.read_csv("Kelompok 5 - Link Artikel LPDP - All.csv")
    df_valid = df_all[df_all["Valid?"] == True].copy().reset_index(drop=True)
    print("Mulai dari awal...")

print(f"Total artikel valid: {len(df_valid)}")

if "Actual_URL" not in df_valid.columns:
    df_valid["Actual_URL"] = None
if "Content" not in df_valid.columns:
    df_valid["Content"] = None


# =========================
# 2. SAFE DECODE URL
# =========================
def safe_decode_url(url):
    for attempt in range(2):
        try:
            result = new_decoderv1(url)
            if result.get("status"):
                return result["decoded_url"]
        except Exception:
            if attempt == 0:
                time.sleep(0.5)
    return None


# =========================
# 3. SCRAPE CONTENT
# =========================
def scrape_content(args):
    idx, url = args
    if url is None or pd.isna(url):
        return idx, None
    try:
        article = Article(url, language="id", request_timeout=8)
        article.download()
        article.parse()
        text = article.text.strip()
        return idx, text if len(text) > 50 else None
    except Exception:
        return idx, None


# =========================
# 4. DECODE URL — SEQUENTIAL
# =========================
need_decode = [
    (i, df_valid.loc[i, "URL"])
    for i in range(len(df_valid))
    if pd.isna(df_valid.loc[i, "Actual_URL"])
]

total_decode = len(need_decode)
print(f"URL yang perlu di-decode: {total_decode}")

AUTOSAVE_EVERY = 50  # autosave tiap 50 URL

for count, (i, url) in enumerate(need_decode, start=1):
    decoded = safe_decode_url(url)
    df_valid.at[i, "Actual_URL"] = decoded

    # Autosave tiap 50 URL
    if count % AUTOSAVE_EVERY == 0 or count == total_decode:
        df_valid.to_csv("hasil_scraping_isi_artikel.csv", index=False)
        terisi = df_valid["Actual_URL"].notna().sum()
        print(f"Decoded {count}/{total_decode} | Total URL terisi: {terisi}")

    time.sleep(0.3)  # jeda kecil agar tidak kena rate limit Google

print("Decode selesai!")


# =========================
# 5. SCRAPE CONTENT — PARALEL
# =========================
need_scrape = [
    (i, df_valid.loc[i, "Actual_URL"])
    for i in range(len(df_valid))
    if pd.isna(df_valid.loc[i, "Content"]) and pd.notna(df_valid.loc[i, "Actual_URL"])
]

total_scrape = len(need_scrape)
print(f"\nArtikel yang perlu di-scrape: {total_scrape}")

SCRAPE_WORKERS = 20
SCRAPE_BATCH = 200

for batch_start in range(0, total_scrape, SCRAPE_BATCH):
    batch = need_scrape[batch_start:batch_start + SCRAPE_BATCH]

    hasil_batch = {}

    with ThreadPoolExecutor(max_workers=SCRAPE_WORKERS) as executor:
        futures = {executor.submit(scrape_content, item): item for item in batch}
        for future in as_completed(futures):
            idx, content = future.result()
            hasil_batch[idx] = content

    # Assign sekaligus setelah semua thread selesai
    for idx, content in hasil_batch.items():
        df_valid.at[idx, "Content"] = content

    df_valid.to_csv("hasil_scraping_isi_artikel.csv", index=False)

    terisi = df_valid["Content"].notna().sum()
    print(f"Scraped {min(batch_start + SCRAPE_BATCH, total_scrape)}/{total_scrape} | Total konten terisi: {terisi}")

print("\nSEMUA ARTIKEL SELESAI")

Resume dari file sebelumnya...
Total artikel valid: 1144
URL yang perlu di-decode: 1144
Decoded 50/1144 | Total URL terisi: 0


KeyboardInterrupt: 